# Import libaries

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))   

from shared import config
from shared.data import get_datasets
from shared.augment import get_augmentation
from shared.evaluate import evaluate


from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")


In [14]:
def build_model(num_classes: int, augmentation: keras.Sequential) -> keras.Model:
    inputs = keras.Input(shape=(config.IMAGE_SIZE, config.IMAGE_SIZE, 3))

    x = augmentation(inputs)
    x = keras.layers.Rescaling(1./255)(x)

    # Convolutional Block 1
    x = keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.25)(x)

    # Convolutional Block 2
    x = keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.25)(x)

    # Convolutional Block 3
    x = keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.25)(x)

    # Fully Connected Layers
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(256, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Summary

In [ ]:
model = build_model(config.NUM_CLASSES, get_augmentation())  
model.summary()                                               

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ aerial_augmentation             │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 56, 56, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,786,564 (98.37 MB)

 Trainable params: 25,785,604 (98.36 MB)

 Non-trainable params: 960 (3.75 KB)

# Compile model

In [20]:
model = build_model(num_classes=config.NUM_CLASSES, augmentation=get_augmentation())

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)


# Callback

In [21]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
]

In [24]:
history = model.fit(
    get_datasets()[0],
    validation_data=get_datasets()[1],
    epochs=15,
    callbacks=callbacks
)

Found 2800 files belonging to 4 classes.
Found 400 files belonging to 4 classes.
Found 2800 files belonging to 4 classes.
Found 400 files belonging to 4 classes.
Epoch 1/15
88/88 ━━━━━━━━━━━━━━━━━━━━ 39s 441ms/step - accuracy: 0.7750 - loss: 0.6289 - val_accuracy: 0.2500 - val_loss: 9.6285 - learning_rate: 4.0000e-06
Epoch 2/15
88/88 ━━━━━━━━━━━━━━━━━━━━ 38s 430ms/step - accuracy: 0.7714 - loss: 0.6214 - val_accuracy: 0.2500 - val_loss: 9.2628 - learning_rate: 4.0000e-06
Epoch 3/15
88/88 ━━━━━━━━━━━━━━━━━━━━ 38s 429ms/step - accuracy: 0.7771 - loss: 0.5944 - val_accuracy: 0.2500 - val_loss: 8.7935 - learning_rate: 4.0000e-06
Epoch 4/15
88/88 ━━━━━━━━━━━━━━━━━━━━ 39s 447ms/step - accuracy: 0.7814 - loss: 0.5815 - val_accuracy: 0.2500 - val_loss: 7.6705 - learning_rate: 1.0000e-06
Epoch 5/15
88/88 ━━━━━━━━━━━━━━━━━━━━ 39s 445ms/step - accuracy: 0.7725 - loss: 0.6138 - val_accuracy: 0.2500 - val_loss: 6.3537 - learning_rate: 1.0000e-06
Epoch 6/15
88/88 ━━━━━━━━━━━━━━━━━━━━ 37s 423ms/step 

# New layers Model

In [36]:
def build_model(num_classes: int, augmentation: keras.Sequential) -> keras.Model:
    inputs = keras.Input(shape=(config.IMAGE_SIZE, config.IMAGE_SIZE, 3))

    x = augmentation(inputs)
    x = keras.layers.Rescaling(1./255)(x)

    # Convolutional Block 1
    x = keras.layers.Conv2D(256,(3,3), activation='relu')(x)
    x = keras.layers.MaxPooling2D((2, 2), padding='valid')(x)
    x = keras.layers.BatchNormalization()(x)

    # Convolutional Block 2
    x = keras.layers.Conv2D(128,(3,3), activation='relu')(x)
    x = keras.layers.MaxPooling2D((2, 2), padding='valid')(x)
    x = keras.layers.BatchNormalization()(x)

    # Convolutional Block 3
    x = keras.layers.Conv2D(128,(3,3), activation='relu')(x)

    # Convolutional Block 4
    x = keras.layers.Conv2D(128,(3,3), activation='relu')(x)

    # Convolutional Block 5
    x = keras.layers.Conv2D(64,(3,3), activation='relu')(x)
    x = keras.layers.MaxPooling2D((2, 2), padding='valid')(x)

    # Fully Connected Layers
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(512, activation='relu')(x)
    x = keras.layers.Dense(256, activation='relu')(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.Dense(16, activation='relu')(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [38]:
model = build_model(num_classes=config.NUM_CLASSES, augmentation=get_augmentation())
model.summary()

Model: "functional_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_20 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ aerial_augmentation             │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_10 (Rescaling)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_39 (Conv2D)              │ (None, 222, 222, 256)  │         7,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_29 (MaxPooling2D) │ (None, 111, 111, 256)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_30          │ (None, 111, 111, 256)  │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_40 (Conv2D)              │ (None, 109, 109, 128)  │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_30 (MaxPooling2D) │ (None, 54, 54, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_31          │ (None, 54, 54, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_41 (Conv2D)              │ (None, 52, 52, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_42 (Conv2D)              │ (None, 50, 50, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_43 (Conv2D)              │ (None, 48, 48, 64)     │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_31 (MaxPooling2D) │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 512)            │    18,874,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,696,468 (75.14 MB)

 Trainable params: 19,695,700 (75.13 MB)

 Non-trainable params: 768 (3.00 KB)

In [40]:
model = build_model(num_classes=config.NUM_CLASSES, augmentation=get_augmentation())

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.00025),   
    loss="categorical_crossentropy",  
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.25, patience=4, min_lr=1e-6
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True
    ),
]

train_ds, val_ds = get_datasets()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
)


Found 2800 files belonging to 4 classes.
Found 400 files belonging to 4 classes.
Epoch 1/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 233s 2s/step - accuracy: 0.4832 - loss: 1.0535 - val_accuracy: 0.2500 - val_loss: 1.4465 - learning_rate: 2.5000e-04
Epoch 2/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.6225 - loss: 0.7705 - val_accuracy: 0.2500 - val_loss: 2.0533 - learning_rate: 2.5000e-04
Epoch 3/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 193s 2s/step - accuracy: 0.7957 - loss: 0.5631 - val_accuracy: 0.4975 - val_loss: 2.1997 - learning_rate: 2.5000e-04
Epoch 4/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 193s 2s/step - accuracy: 0.8504 - loss: 0.4266 - val_accuracy: 0.2650 - val_loss: 6.4176 - learning_rate: 2.5000e-04
Epoch 5/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 192s 2s/step - accuracy: 0.8643 - loss: 0.3894 - val_accuracy: 0.2500 - val_loss: 6.6402 - learning_rate: 2.5000e-04
Epoch 6/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 196s 2s/step - accuracy: 0.9204 - loss: 0.2226 - val_accuracy: 0.2625 - val_loss: 5.1253 - learning_rate: 